# 03 · Ask a question

The frozen pipeline on a new question: BM25 + Vietnamese_Embedding → rank fusion → bge-reranker-v2-m3 → top 5 → LLM with citations.

Needs: the chunk vectors from `scripts/run_models.py` (`data/index/vn-embedding.npz`), torch + sentence-transformers (the two models load from the Hugging Face cache), and an LLM in `.env` (or Ollama running).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.rag.dataset import load_chunks
from src.rag.llm import from_env
from src.rag.pipeline import LiveRetriever, ask

retriever = LiveRetriever(load_chunks())   # loads the embedder and the reranker once
llm = from_env('api')                      # or from_env('local') for qwen3.5 through Ollama

In [ ]:
def show(question, options=None):
    out = ask(question, retriever, llm, options)
    print(out['answer'])
    print()
    for n, src in out['sources'].items():
        print(f'[{n}] {src}')
    if out['refused']:
        print('(refused: the retrieved excerpts do not contain the answer)')

show('Vắng thi có phép thì nhận điểm gì và khi nào được thi lại?')

A question the documents cannot answer: the assistant should refuse rather than guess.

In [ ]:
show('Thư viện trường mở cửa đến mấy giờ?')

Your own question:

In [ ]:
show('Điểm rèn luyện bao nhiêu thì bị cảnh báo học vụ?')